# Space Debris Analysis Project

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

## 1. Data Loading & Initial Exploration
### Extracting and Saving the File

#### Note: HTTP Response Codes & Methods

**Response Codes Reference**

* **1xx**: Informational (request received, continuing process)
* **200**: Success
* **3xx**: Redirection (further action needed)
* **401**: Unauthorized
* **403**: Forbidden
* **404**: Not Found

**Diagnostic Code Checks**

* `r.status_code`
* `r.request.headers`
* `r.request.body`
* `r.headers`

**When to Use Different Response Methods**

* **`.json()`** - Use when API returns JSON data
  * Most modern REST APIs (GitHub, weather, etc.)
  * Common for web services and public APIs

* **`.text`** - Use for plain text responses
  * HTML pages
  * Raw string data

* **`.content`** - Use for binary data
  * Images
  * File downloads
  * Returns raw bytes that you can save to a file

In [0]:
%skip
# data extracted url
url = "https://celestrak.org/pub/satcat.csv"

# path where you want to save the file
path = "/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/data/raw"
# path = "/Workspace/Users/guruvendra47@gmail.com/space-debris-project/space_debris_raw.csv" when you want code simpler and dont want filname seperated variable
# file name which you want to give
filename = "space_debris_raw.csv"
# combine full path with your filename
full_path = f"{path}/{filename}"


try:
    # getting data from url
    response = requests.get(url)

    # check if it is success or failed
    if response.status_code == 200:
        with open(full_path, "wb") as f:
            f.write(response.content)
            print(f"File saved successfully to: {full_path}")
    else:
        print(f"Failed to download. Status code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")


In [0]:
path = "/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/data/raw/space_debris_raw.csv"
df = pd.read_csv(path)
df

### Dataset Overview

In [0]:
%skip
# Check your full dataset
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

display(df)


### Basic Pandas Operations
- .head()
- .tail()
- .shape
- .columns
- .info()
- .describe()
- .rename(column=oldcolumnname, newcolumnname)

In [0]:
# head
df.head(10)

In [0]:
df.tail(10)

In [0]:
df.shape

In [0]:
# df.info()
df.dtypes

In [0]:
df.columns

#### Notes
**Column Headers**

* **`rename()`**: Changes header titles.

**Row Values**

* **`replace()`**: Use for quick code without a loop.
* **`map().fillna()`**: Use with a loop for max speed.

In [0]:
# rename the columns
rename_mapping={
        "OBJECT_NAME": "ObjectName",
        "OBJECT_ID": "ObjectID",
        "NORAD_CAT_ID": "CatalogID",
        "OBJECT_TYPE": "ObjectType",
        "OPS_STATUS_CODE": "OperationalStatus",
        "OWNER": "Owner",
        "LAUNCH_DATE": "LaunchDate",
        "LAUNCH_SITE": "LaunchSite",
        "DECAY_DATE": "DecayDate",
        "PERIOD": "OrbitalPeriodMin",
        "INCLINATION": "InclinationDegrees",
        "APOGEE": "MaxAltitudeKM",
        "PERIGEE": "MinAltitudeKM",
        "RCS": "RadarSizeSQM",
        "DATA_STATUS_CODE": "DataStatus",
        "ORBIT_CENTER": "OrbitCenter",
        "ORBIT_TYPE": "OrbitState",
    }

df = df.rename(columns=rename_mapping)

print(df.columns.tolist())

In [0]:
df.head(4)

### Understanding the Data
#### Space Debris Dataset Column Descriptions

* **`satellite_name`**: This columns contain names of satellite, rocket stage, or space debris.
* **`satellite_id`**: This is Unique No assigned to satellite, rocket stage, or space debris in International Designator (COSPAR ID) format as `YYYY-NNNAA` (Launch Year, Launch Number, Piece of launch).
* **`norad_cat_id`**: Unique tracking catalog number assigned by USSPACECOM/NORAD.
* **`satellite_type`**: Classification of the object:
  * `PAY`: Payload (active or inactive satellite).
  * `R/B`: Rocket Body (spent upper stage left in orbit).
  * `DEB`: Debris (fragmentation or dropped hardware).
* **`operation_status_code`**: Operational status indicator:
  * `+`: Operational/Active.
  * `-`: Non-operational/Inactive.
  * `P` : Partially Operational / Standby
  * `D`: Decayed (re-entered Earth's atmosphere).
  * `NaN`: Unknown or unassigned status.
* **`country`**: Country, space agency, or commercial owner.
* **`launch_date`**: Date the object was launched into space.
* **`launch_location`**: launch site/spaceport.
* **`decay_date`**: Date in which the object re-entered Earth's atmosphere. Missing values (`NaN`/`NaT`) mean the object is still in orbit.
* **`period_time_min`**: Orbital period in minutes (time required to complete one full orbit around Earth).
* **`Inclination`**: Angle (in degrees) between the orbital plane and Earth's equator.
* **`highest_altitude_km`**: Highest point of the orbit from Earth's surface (Apogee in km).
* **`lowest_altitude_km`**: Lowest point of the orbit from Earth's surface (Perigee in km).
* **`radar_cross_section`**: Radar Cross Section (\text{m}^2), representing the reflective physical size detected by radar.
* **`data_status_code`**: Administrative code indicating tracking data reliability.
* **`orbit_center`**: Body being orbited (`EA` = Earth) etc.
* **`orbit_type`**: Orbital state (`ORB` = Currently orbiting, `IMP` = Impacted/Decayed).



## 2. Data Wrangling (Cleaning)

### Step 1: Finding and Removing Duplicates
1. Identify duplicate rows  in the dataset.
2. Use suitable techniques to remove duplicate rows and verify the removal.
3. Summarize how to handle missing values appropriately.
4. Use ConvertedCompYearly to normalize compensation data.

In [0]:
# df.duplicated().value_counts()
df.duplicated().sum()

# to see duplicate
df[df.duplicated()]

# df.duplicated(subset=columnslist, keep=False) in order to check how many same values are there over the data 

# Removing the Duplicates
#df.drop_duplicates()

#### Note: No Duplicate Values Found

### Step 2: Removing Whitespaces
##### Note: 
- We only check string columns (dtype == 'object') because numeric columns (int, float) cannot contain leading/trailing whitespaces.

In [0]:
# step 1: Remove whitespaces

# Before: Count rows with whitespaces in each column
print("=== Before: Whitespace Analysis ===")
for i in df.columns:
    if df[i].dtype == 'object':
        wht_spc = (df[i].notna()) & (df[i] != df[i].str.strip())
        cnt_wht = wht_spc.sum()
        if cnt_wht > 0:
            print(f"{i}: {cnt_wht}")
        else:
            print(f"{i}: {cnt_wht}")

print("\nRemoving....")
for i in df.columns:
    if df[i].dtype == 'object':
        df[i] = df[i].str.strip()

# After: Count rows with whitespaces in each column
print("\n=== After: Whitespace Analysis ===")
for i in df.columns:
    if df[i].dtype == 'object':
        wht_spc = (df[i].notna()) & (df[i] != df[i].str.strip())
        cnt_wht = wht_spc.sum()
        if cnt_wht > 0:
            print(f"{i}: {cnt_wht} ")
        else:
            print(f"{i}: {cnt_wht}")


#### Outlier Analysis (Optional)
-  For this data, we do not need to remove outliers as the extreme values are real data.

**Box Plot**: 
Use when you are analyzing a single numerical variable at a time (or comparing a single numerical variable across categories, like MaxAltitudeKM grouped by ObjectType). It shows distribution, quartiles, and statistical outliers cleanly (e.g., `OrbitalPeriodMin` or `RadarSizeSQM`).

**Scatter Plot**: 
Use when you are comparing two (or more) numerical variables against each other to see relationships, trends, correlations, and bivariate outliers (e.g., `MinAltitudeKM` vs. `MaxAltitudeKM`).



In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# select only the numeric columns
num_col = df.select_dtypes(include=["number"])

# check lower quartile and upper quartile
Q1 = num_col.quantile(0.25)
Q3 = num_col.quantile(0.75)
IQR = Q3 - Q1

# checking lower and upper values
lower_value = Q1 - 1.5 * IQR
upper_value = Q3 + 1.5 * IQR

# Converting columns to Boolean so it will return True where an outlier exists
otlr = (num_col < lower_value) | (num_col > upper_value)

# Count total outliers per column
cnt_otlr = otlr.sum()
print("outlier_counts\n", cnt_otlr)

# if you want to see rows with outlier
rws_otlr = df[otlr.any(axis=1)]
print("Find outlier in Columns")
print(rws_otlr)
print("")

rws_otlr.head()

#  why Box plot not scatter plot because individual value checking not comparing for visualization
plt.figure(figsize=(18, 12))

plt.subplot(3,3,1)
sns.boxplot(y=df["OrbitalPeriodMin"])
plt.title('OrbitalPeriodMin')

plt.subplot(3,3,2)
sns.boxplot(y=df["InclinationDegrees"])
plt.title('InclinationDegrees')

plt.subplot(3,3,3)
sns.boxplot(y=df["MaxAltitudeKM"])
plt.title('MaxAltitudeKM')

plt.subplot(3,3,4)
sns.boxplot(y=df["MinAltitudeKM"])
plt.title('MinAltitudeKM')

plt.subplot(3,3,5)
sns.boxplot(y=df["RadarSizeSQM"])
plt.title('RadarSizeSQM')

plt.subplot(3,3,6)
sns.boxplot(y=df["CatalogID"])
plt.title('CatalogID')

plt.suptitle('Analysing Outlier', fontsize=16)
plt.tight_layout()
plt.show()

### Step 3: Changing Data Types
- check which columns data type is wrong 
- change into "object", "int64", float64", "datatime64[ns]"
- we found that LaunchDate and DecayDate both columns dtypes need to change from object to Datetime

In [0]:
# check which columns has wrong data type
print("=== Before ===")
display(df.info())

print("\nChanging....")
df["LaunchDate"] = pd.to_datetime(df["LaunchDate"],format="mixed", errors= "coerce")
df["DecayDate"] = pd.to_datetime(df["DecayDate"],format="mixed", errors="coerce")

print("\n\n=== After ===")
print(df.dtypes)


### Step 4: Finding and Handling Missing Values
1. Identifying the missing values and converting in percentage for better view
2. Visualize missing values using a heatmap
3. Count the number of missing rows for a specific column (.value_counts())
4. Identifing the most frequent or more time repeated value in a spcific column
5. Taking decision to drop the rows or column or fill with mean, median, .idxmax()(most repeated value), .ffill()(forward-fill is filling with next value)
6. Visualize the distribution of a column after imputationF

In [0]:
# identifing the missing values
msgvalue = df.isnull().sum()

# In percentage for better view and rounded to 2 decimal 
prt = (msgvalue / len(df)*100).round(2)

# putting side by side for better view
dic = pd.DataFrame({"Missingvalues": msgvalue, "Percentage (%)":prt})
print(dic)

#### Note: Missing Values Summary
- OperationalStatus
- DecayDate
- OrbitalPeriodMinutes
- InclinationDegrees
- ApogeeKM
- PerigeeKM 
- RadarCrossSectionSQM
- DataStatusCode


#### Column: OperationalStatus
#### Column: OperationalStatus

Missing values occur because space tracking agencies only assign status codes to active/inactive satellites. used upper rocket bodies and debris do not have operational missions, so their status is naturally left blank so we filling that as `"Unknown"`.

In [0]:
# fillna NaN with Unknown for operation status code
df["OperationalStatus"] = df["OperationalStatus"].fillna("Unknown")
print(df.isnull().sum())

#### Column: DecayDate
#### Column: DecayDate
 - A missing `DecayDate` (`NaT`) means the object has not re-entered Earth's atmosphere and is still currently in orbit. We keep these values empty to maintain the `datetime64` data type so date calculations do not break. In Power BI, null values will be used to filter and render all active objects around Earth.

In [0]:

# DecayDate kept as NaT - missing values mean the object is still in space. Since it's not a string, I left NaT rather than changing it to "in space orbit".


#### Columns: Orbital Parameters

**Columns:** OrbitalPeriodMin, InclinationDegrees, MaxAltitudeKM, MinAltitudeKM

- Orbital parameters cannot be filled with zero (0 minutes = invalid orbit).
- In this case, we have to fill NaN values with median values, grouped by `ObjectType` and filtered by `OrbitCenter`.
- This ensures Earth-orbiting payloads use Earth payload medians, etc.

In [0]:
df["OrbitCenter"].unique()

In [0]:

# columns : period_time_min, Inclination, highest_altitude_km, lowest_altitude_km

cols = ["OrbitalPeriodMin", "InclinationDegrees", "MaxAltitudeKM", "MinAltitudeKM"]


orb = df["OrbitCenter"].unique()
# filtering data
for i in orb:
    mask = df["OrbitCenter"] == i
    # grouping the data
    for j in cols:
        df.loc[mask, j] = df.loc[mask, j].fillna(df[mask].groupby("ObjectType")[j].transform("median"))

print("\n=== Remaining Missing Values ===")
print(df.isnull().sum())

#### Note: Remaining Nulls values After Group Median
- After filtering with OrbitCenter and groupby ObjectType, 67 nulls remain in the "OrbitalPeriodMinutes", "OrbitalTiltDegrees", "HighestPointKm", and "LowestPointKm" columns because the mean could not be calculated because these columns value is null.
- As orbital data, making group medians NaN. We fill them with -1 (sentinel value) to keep numeric dtypes, preserve row count.

In [0]:
# 67 nan we will with sentinel value -1 as it non-earth orbit

df[cols] = df[cols].fillna(-1)
print(df.isnull().sum())


#### Column: RadarCrossSectionSQM
#### Note: RadarSizeSQM Null Handling

- Radar Cross Section (RCS) is missing for objects that are too small or far away to be measured by ground radar. We fill missing values with `0.0` to treat them as untracked physical sizes without breaking numeric models.

In [0]:
# radar_cross_section column
df["RadarSizeSQM"] = df["RadarSizeSQM"].fillna(0.0)
print(df.isnull().sum())


#### Column: DataStatus 

- According to space tracking documentation (CelesTrak SATCAT Documentation), 
- the official meanings of DataStatusCodes are:
   - NIE: No Initial Elements (Sensors detected the object at launch, but stable initial orbital calculations could not be established)
   - NEA: No Elements Available (Tracking elements are missing or discontinued)
   - NCE: No Current Elements (Historical tracking exists, but current active updates are unassigned)
   - NaN (Blank): Nominal Tracking or Active Elements Available (The standard baseline state for cataloged objects)

- Strategy: Replace codes with descriptive labels and fill NaN with "Active Tracking"

In [0]:
# 98 percentage is active tracking data which is null 

stcd = {"NIE":"No Initial Elements", "NEA":"No Elements Available", "NCE": "No Current Elements"}
df["DataStatus"] = df["DataStatus"].replace(stcd)
df["DataStatus"] = df["DataStatus"].fillna("Active Tracking")
print(df.isnull().sum())
df

## 3. Data Standardization

*Converts short codes, acronyms, and abbreviations into human-readable text for Power BI visuals and reports.*

#### Columns to Standardize:

- ObjectType 
- OperationalStatus
- Owner 
- LaunchSite
- OrbitCenter
- OrbitState
- DataStatus.



### Column Modification Methods

**Column Headers**

* **`rename()`**: Changes header titles.

**Row Values**

* **`replace()`**: Use for quick code without a loop.
* **`map().fillna()`**: Use with a loop for max speed.

### .map() vs .replace()

**Use `.map()`:**
- When remapping an entire column into a new scale
- Unlisted values automatically turn to NaN
- Best for complete transformations

**Use `.replace()`:**
- When fixing a couple of specific values
- Everything else stays as-is
- Best for targeted replacements

### Step 1: ObjectType & OperationalStatus

In [0]:

# No 1
# Column: ObjectType
dic = {"R/B":"Rocket Body", "PAY":"Payload", "DEB": "Debris"}
df["ObjectType"] = df["ObjectType"].replace(dic)
df.head()

# No2
# Column: OperationalStatus
dic = {"D": "Decayed (D)", "+": "Operational (+)", "-":"Non-Operational (-)", "P": "Partially Operational (P)"  }
df["OperationalStatus"] = df["OperationalStatus"].replace(dic)
df



### Step 2: Owner

In [0]:
# check Unique Country code
df["Owner"].unique()

In [0]:
owners = {
    "AB": "ARABSAT (Arab Satellite)",
    "AC": "Asia-Pacific Space Cooperation",
    "ABS": "ABS (Asia Broadcast Satellite)",
    "ALG": "Algeria",
    "ANG": "Angola",
    "ARGN": "Argentina",
    "ARM": "Armenia",
    "ASRA": "AsiaSat Telecommunications",
    "AUS": "Australia",
    "AZER": "Azerbaijan",
    "BEL": "Belgium",
    "BELA": "Belarus",
    "BGD": "Bangladesh",
    "BHR": "Bahrain",
    "BHUT": "Bhutan",
    "BOL": "Bolivia",
    "BRAZ": "Brazil",
    "BUL": "Bulgaria",
    "BWA": "Botswana",
    "CA": "Canada",
    "CHBZ": "China / Brazil (CBERS)",
    "CHLE": "Chile",
    "CIS": "Russia / Former USSR",
    "COL": "Colombia",
    "CRI": "Costa Rica",
    "CZCH": "Czech Republic",
    "DEN": "Denmark",
    "DJI": "Djibouti",
    "ECU": "Ecuador",
    "EGYP": "Egypt",
    "ESA": "European Space Agency",
    "ESRO": "European Space Research Org",
    "EST": "Estonia",
    "ETH": "Ethiopia",
    "EUME": "EUMETSAT",
    "EUTE": "EUTELSAT",
    "FGER": "West Germany (Former)",
    "FIN": "Finland",
    "FR": "France",
    "FRIT": "France / Italy",
    "GER": "Germany",
    "GHA": "Ghana",
    "GLOB": "Globalstar (Commercial)",
    "GREC": "Greece",
    "GRSA": "South Africa",
    "GUAT": "Guatemala",
    "HRV": "Croatia",
    "HUN": "Hungary",
    "IM": "Inmarsat (Commercial)",
    "IND": "India",
    "INDO": "Indonesia",
    "IRAN": "Iran",
    "IRAQ": "Iraq",
    "IRL": "Ireland",
    "ISRA": "Israel",
    "ISS": "International Space Station",
    "IT": "Italy",
    "ITSO": "INTELSAT (Commercial)",
    "JOR": "Jordan",
    "JPN": "Japan",
    "KAZ": "Kazakhstan",
    "KEN": "Kenya",
    "KWT": "Kuwait",
    "LAOS": "Laos",
    "LKA": "Sri Lanka",
    "LTU": "Lithuania",
    "LUXE": "Luxembourg",
    "MA": "Morocco",
    "MALA": "Malaysia",
    "MCO": "Monaco",
    "MDA": "Moldova",
    "MEX": "Mexico",
    "MMR": "Myanmar (Burma)",
    "MNE": "Montenegro",
    "MNG": "Mongolia",
    "MUS": "Mauritius",
    "NATO": "NATO",
    "NETH": "Netherlands",
    "NICO": "New ICO",
    "NIG": "Nigeria",
    "NKOR": "North Korea",
    "NOR": "Norway",
    "NPL": "Nepal",
    "NZ": "New Zealand",
    "O3B": "O3b Networks / SES",
    "ORB": "Orbcomm (Commercial)",
    "PAKI": "Pakistan",
    "PERU": "Peru",
    "POL": "Poland",
    "POR": "Portugal",
    "PRC": "China",
    "PRY": "Paraguay",
    "QAT": "Qatar",
    "RASC": "RASCOMSTAR-QAF",
    "ROC": "Taiwan",
    "ROM": "Romania",
    "RP": "Philippines",
    "RWA": "Rwanda",
    "SAFR": "South Africa",
    "SAUD": "Saudi Arabia",
    "SDN": "Sudan",
    "SEAL": "Sea Launch",
    "SEN": "Senegal",
    "SES": "SES S.A. (Commercial)",
    "SGJP": "Singapore / Japan",
    "SING": "Singapore",
    "SKOR": "South Korea",
    "SLB": "Solomon Islands",
    "SPN": "Spain",
    "STCT": "Singapore / Taiwan (ST-1)",
    "SVK": "Slovakia",
    "SVN": "Slovenia",
    "SWED": "Sweden",
    "SWTZ": "Switzerland",
    "TBD": "Unassigned / To Be Determined",
    "THAI": "Thailand",
    "TMMC": "Turkmenistan / Monaco",
    "TUN": "Tunisia",
    "TURK": "Turkey",
    "UAE": "United Arab Emirates",
    "UGA": "Uganda",
    "UK": "United Kingdom",
    "UKR": "Ukraine",
    "URY": "Uruguay",
    "US": "United States",
    "USBZ": "United States / Brazil",
    "VAT": "Vatican City",
    "VENZ": "Venezuela",
    "VTNM": "Vietnam",
    "ZWE": "Zimbabwe",
}

df["Owner"] = df["Owner"].replace(owners)

df

### Step 3: LaunchSite 


In [0]:
df["LaunchSite"].unique()

#### Note: Power BI Reminder
- In Power BI, make sure you document what the launch site codes represent.

In [0]:
sitemapping = {
    "AFETR": "United States (AFETR)",
    "AFWTR": "United States (AFWTR)",
    "CAS": "Spain (CAS)",
    "DLS": "Russia (DLS)",
    "ERAS": "United States (ERAS)",
    "FRGUI": "French Guiana (FRGUI)",
    "HGSTR": "Algeria (HGSTR)",
    "JJSLA": "South Korea (JJSLA)",
    "JSC": "China (JSC)",
    "KODAK": "United States (KODAK)",
    "KSCUT": "Japan (KSCUT)",
    "KWAJ": "Marshall Islands (KWAJ)",
    "KYMSC": "Russia (KYMSC)",
    "NSC": "South Korea (NSC)",
    "PLMSC": "Russia (PLMSC)",
    "RLLB": "New Zealand (RLLB)",
    "SCSLA": "China (SCSLA)",
    "SEAL": "International (SEAL)",
    "SEMLS": "Iran (SEMLS)",
    "SMTS": "Iran (SMTS)",
    "SNMLP": "Kenya (SNMLP)",
    "SRILR": "India (SRILR)",
    "SUBL": "International (SUBL)",
    "SVOBO": "Russia (SVOBO)",
    "TAISC": "China (TAISC)",
    "TANSC": "Japan (TANSC)",
    "TYMSC": "Kazakhstan (TYMSC)",
    "VOSTO": "Russia (VOSTO)",
    "WLPIS": "United States (WLPIS)",
    "WOMRA": "Australia (WOMRA)",
    "WRAS": "United States (WRAS)",
    "WSC": "China (WSC)",
    "XICLF": "China (XICLF)",
    "YAVNE": "Israel (YAVNE)",
    "YSLA": "China (YSLA)",
    "YUN": "North Korea (YUN)",
}

df["LaunchSite"] = df["LaunchSite"].map(sitemapping).fillna(df["LaunchSite"])
# or
df["LaunchSite"] = df["LaunchSite"].replace(sitemapping)
df.head(20)

### Step 4: OrbitCenter

In [0]:
df["OrbitCenter"].unique()

#### Note: In Power BI 

- Create 2 quick fields In Power BI one for top slicers (Target_Category: Planet, Orbit) and one for clean hover tooltips (Target_Name: Earth (EA)).

In [0]:
orbname = {
    # Planets & Celestial Bodies
    "ME": "Mercury (ME)",
    "VE": "Venus (VE)",
    "EA": "Earth (EA)",
    "MA": "Mars (MA)",
    "JU": "Jupiter (JU)",
    "SA": "Saturn (SA)",
    "UR": "Uranus (UR)",
    "NE": "Neptune (NE)",
    "PL": "Pluto (PL)",
    "SU": "Sun (SU)",
    "MO": "Moon (MO)",
    "CO": "Comet (CO)",
    # Systems & Lagrange Points
    "EM": "Earth-Moon System (EM)",
    "SS": "Solar System (SS)",
    "AS": "Asteroids (AS)",
    "EL": "Earth-Moon Lagrange (EL)",
    "EL1": "Earth-Moon L1 (EL1)",
    "EL2": "Earth-Moon L2 (EL2)",
    # Specific High-Profile Space Stations
    "25544": "ISS (25544)",
    "28358": "Tiangong Target (28358)",
    "48274": "Tianhe Core (48274)",
}

df["OrbitCenter"] = df["OrbitCenter"].replace(orbname)
df

### Step 5: OrbitState

In [0]:
df["OrbitState"].unique()

In [0]:
obste = {
    # Existing Array Codes
    "IMP": "Impact (IMP)",
    "ORB": "Orbiter (ORB)",
    "LAN": "Lander (LAN)",
    "DOC": "Dock / Rendezvous (DOC)",
    # Other CelesTrak Mission Types
    "FLY": "Flyby (FLY)",
    "SAMP": "Sample Return (SAMP)",
    "CREW": "Crewed (CREW)",
    "CARG": "Cargo / Resupply (CARG)",
    "DEP": "Deployer (DEP)",
    "REL": "Relay (REL)",
}

df["OrbitState"] = df["OrbitState"].replace(obste)
df

#### Note: Advanced Data Transformation Methods
**`.explode()`** - Turns lists into individual rows, temporarily duplicating row values so pandas can evaluate every item separately.
- Must be assigned to a variable (does not save changes automatically).

**`.melt()`**
- Converts a DataFrame from Wide Format (side-by-side columns) into Long Format (stacked rows). 
- **Example** of what I understood: think of months [January, February, March] as columns, then making one column as "Month" and adding all months like January, February, March as rows.
- Unpivots specified columns into key-value pairs (variable and value), repeating non-melted identifier columns for each row.
- Must be assigned to a variable (does not save changes automatically).


## 4. Data Normalization / Feature Scaling

#### Note: Feature Scaling Not Applied 
- Feature scaling is not applied to this project as we not going to do any Machine learning part.

### When to Use Feature Scaling (For ML Modeling)

*Rescales continuous numerical features with large variations (e.g., altitudes reaching 36,000km vs. orbital periods of 90minutes) so algorithms treat them equally.*

* **`OrbitalPeriodMinutes`**
* **`InclinationDegrees`**
* **`ApogeeKM`**
* **`PerigeeKM`**
* **`RadarCrossSectionSQM`**



#### Note: Scaling Methods for ML

* **`RobustScaler`**
* **When to use:** When data has **heavy/extreme outliers**.
* **Formula:**
$$x_{\text{scaled}} = \frac{x - \text{Median}}{\text{IQR}}$$


*Where IQR = Q3 (75th percentile) - Q1 (25th percentile)*


* **`MinMaxScaler`**
* **When to use:** When we need a **values in [0, 1] range** and have **no outliers**.
* **Formula:**
$$x_{\text{scaled}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}$$




* **`StandardScaler`**
* **When to use:** When data follows a **bell curve (normal distribution)** with **few or no outliers**.
* **Formula:**
$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$


mean is $${\mu}$$ 
standard deviation is $$\sigma$$



---

### Code

```python
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

```

## 5. Data Binning (Categorical Grouping)

#### Note:

**When to Bin in Python**

* If you're building **machine learning models** → bin in Python
* If you're making **charts in Python** (matplotlib, seaborn, Plotly) → bin in Python
* If you have **fixed business rules** that never change → bin in Python

---

**When NOT to Bin (Let Power BI Do It)**

* If your **final dashboard is in Power BI** → don't bin in Python
* If you want **smaller file sizes** → don't bin in Python
* If users need to **change bin ranges** in the report → don't bin in Python
* If you need **exact numbers later** → don't bin in Python

#### Column: MaxAltitudeKM
* **-1 - 2000:** Low Earth Orbit (LEO)
* **2000 - 35785:** Medium Earth Orbit (MEO)
* **35785 - 36000:** Geostationary Earth Orbit (GEO)
* **36000 - np.inf:** High Earth Orbit (HEO)

In [0]:

# 1. Orbital Altitude Bins (ApogeeKM)
# low -1 to 2000, medium 2000 to 35785, geo 35785 to 36000, Heo 36000 to infinity (float('inf))
orbit_bins = [-1, 2000, 35785, 36000, np.inf]
orbit_labels = ['Low Earth Orbit', 'Medium Earth Orbit', 'Geostationary Orbit', 'High Earth Orbit']
df['OrbitClass'] = pd.cut(df['MaxAltitudeKM'], bins=orbit_bins, labels=orbit_labels)

df

#### Column: RadarCrossSectionSQM
* **-1 - 0.1**: Small
* **0.1 - 1.0**: Medium
* **1.0 - np.inf**: Large

In [0]:
# Column: RadarCrossSectionSQM

size_bins = [-1, 0.1, 1.0, np.inf]
size_labels = ['Small', 'Medium', 'Large']
df['RadarSize'] = pd.cut(df['RadarSizeSQM'], bins=size_bins, labels=size_labels)
df.head(4)

#### Column: LaunchDate
* **1980-1900:** *19th Century*
* **1900–2000:** *20th century*
* **2000–Present:** *21st century*

In [0]:
# Column: LaunchDate 

launch_bins = [1800, 1900, 2000, 2035]
launch_labels = ['19th Century', '20th Century', '21st Century']
df['LaunchEra'] = pd.cut(df['LaunchDate'].dt.year, bins=launch_bins, labels=launch_labels)

## 6. Creating Indicator Variables

### Binary Flags for Power BI

*Creates binary (1 or 0) flags to easily isolate specific conditions in Power BI measures or ML features.*

* **`IsDecayed`:** `1` if `DecayDate` is present (or `OrbitState == "IMP"`), `0` if still in orbit.
* **`IsActive`:** `1` if `OperationalStatus == "+"` (or `ObjectType == "PAY"` and `DecayDate` is null), `0` otherwise.
* **`IsDebris`:** `1` if `ObjectType == "DEB"`, `0` otherwise.
* **`IsRocketBody`:** `1` if `ObjectType == "R/B"`, `0` otherwise.



In [0]:
import numpy as np

# 1. IsDecayed: 1 if DecayDate is present or OrbitState is 'IMP', 0 if still in orbit
df["IsDecayed"] = np.where(df["DecayDate"].notna() | (df["OrbitState"] == "IMP"), 1, 0)

# 2. IsActive: 1 if OperationalStatus is '+' or (ObjectType is 'PAY' and DecayDate is NaT), 0 otherwise
df["IsActive"] = np.where((df["OperationalStatus"] == "+") | ((df["ObjectType"] == "PAY") & (df["DecayDate"].isna())), 1, 0)

# 3. IsDebris: 1 if ObjectType is 'DEB', 0 otherwise
df["IsDebris"] = np.where(df["ObjectType"] == "DEB", 1, 0)

# 4. IsRocketBody: 1 if ObjectType is 'R/B', 0 otherwise
df["IsRocketBody"] = np.where(df["ObjectType"] == "R/B", 1, 0)

# Display the newly created binary columns to verify
display(df[["ObjectName", "ObjectType", "IsDecayed", "IsActive", "IsDebris", "IsRocketBody"]].head(10))

## 7. Exploratory Data Analysis

### Statistical Summary (Lab 12)
*Descriptive statistics for all numerical variables*

In [0]:
# Get comprehensive statistical summary of all numerical columns
print("=" * 80)
print("STATISTICAL SUMMARY OF NUMERICAL VARIABLES")
print("=" * 80)

# Select only numerical columns to avoid Arrow serialization issues
numeric_df = df.select_dtypes(include=['float64', 'int64'])

# Display describe() output
stats_summary = numeric_df.describe()
display(stats_summary)

# Additional insights
print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

print(f"\n📊 Dataset Size: {len(df):,} space objects")
print(f"\n🔢 Numerical Variables Analyzed: {len(stats_summary.columns)}")

print("\n📈 Altitude Statistics (km):")
print(f"   • Mean Max Altitude: {df['MaxAltitudeKM'].mean():.2f} km")
print(f"   • Median Max Altitude: {df['MaxAltitudeKM'].median():.2f} km")
print(f"   • Range: {df['MaxAltitudeKM'].min():.2f} to {df['MaxAltitudeKM'].max():.2f} km")

print("\n⏱️ Orbital Period Statistics (minutes):")
print(f"   • Mean Period: {df['OrbitalPeriodMin'].mean():.2f} min")
print(f"   • Median Period: {df['OrbitalPeriodMin'].median():.2f} min")
print(f"   • Standard Deviation: {df['OrbitalPeriodMin'].std():.2f} min")

print("\n📡 Radar Size Statistics (m²):")
valid_radar = df[df['RadarSizeSQM'] > 0]['RadarSizeSQM']
print(f"   • Mean: {valid_radar.mean():.4f} m²")
print(f"   • Median: {valid_radar.median():.4f} m²")
print(f"   • Objects with radar data: {len(valid_radar):,}")

### Distribution Analysis (Lab 13)
*Testing for normal distribution - skewness and kurtosis*

In [0]:
from scipy.stats import skew, kurtosis
import pandas as pd

# Select numerical columns for distribution analysis
numerical_cols = ['OrbitalPeriodMin', 'InclinationDegrees', 'MaxAltitudeKM', 
                  'MinAltitudeKM', 'RadarSizeSQM']

# Calculate skewness and kurtosis for each numerical column
print("=" * 80)
print("DISTRIBUTION SHAPE ANALYSIS")
print("=" * 80)

distribution_analysis = []

for col in numerical_cols:
    # Filter out invalid values (like -1 sentinel values)
    valid_data = df[df[col] > 0][col].dropna()
    
    if len(valid_data) > 0:
        skewness = skew(valid_data)
        kurt = kurtosis(valid_data)
        
        # Interpret skewness
        if abs(skewness) < 0.5:
            skew_interp = "Approximately Symmetric"
        elif skewness > 0.5:
            skew_interp = "Right-skewed (positive)"
        else:
            skew_interp = "Left-skewed (negative)"
        
        # Interpret kurtosis
        if abs(kurt) < 0.5:
            kurt_interp = "Normal (mesokurtic)"
        elif kurt > 0.5:
            kurt_interp = "Heavy-tailed (leptokurtic)"
        else:
            kurt_interp = "Light-tailed (platykurtic)"
        
        distribution_analysis.append({
            'Variable': col,
            'Skewness': f"{skewness:.3f}",
            'Skew_Interpretation': skew_interp,
            'Kurtosis': f"{kurt:.3f}",
            'Kurt_Interpretation': kurt_interp,
            'Valid_N': len(valid_data)
        })

# Display as DataFrame
dist_df = pd.DataFrame(distribution_analysis)
print("\n")
display(dist_df)

print("\n" + "=" * 80)
print("INTERPRETATION GUIDE")
print("=" * 80)
print("""\n📊 Skewness:
   • Close to 0 (-0.5 to 0.5): Symmetric distribution
   • > 0.5: Right-skewed (long tail on right, mean > median)
   • < -0.5: Left-skewed (long tail on left, mean < median)

📊 Kurtosis:
   • Close to 0 (-0.5 to 0.5): Normal distribution shape
   • > 0.5: Heavy tails (more outliers than normal)
   • < -0.5: Light tails (fewer outliers than normal)
""")

print("\n💡 KEY FINDINGS:")
print("   • Most space debris variables show RIGHT-SKEWED distributions")
print("   • This is typical for orbital data - many objects cluster at lower")
print("     altitudes/periods, with fewer extreme high-value outliers")
print("   • Heavy tails indicate presence of extreme orbital configurations")

### EDA Analysis: Categorical Variables - Value Counts

In [0]:
# Top 10 Object Types
print("=== Object Type Distribution ===")
print(df['ObjectType'].value_counts())
print(f"\nTotal: {df['ObjectType'].value_counts().sum()}\n")

# Operational Status
print("=== Operational Status Distribution ===")
print(df['OperationalStatus'].value_counts())
print(f"\nTotal: {df['OperationalStatus'].value_counts().sum()}\n")

# Top 15 Owners/Countries
print("=== Top 15 Owners/Countries ===")
print(df['Owner'].value_counts().head(15))
print(f"\nTotal unique owners: {df['Owner'].nunique()}")

### EDA Analysis: Grouping by Attributes

In [0]:
# Group by Object Type and calculate the mean for numerical columns
df_group_type = df.groupby('ObjectType', as_index=False)[['OrbitalPeriodMin', 'MaxAltitudeKM', 'MinAltitudeKM']].mean()

# Display the grouped results
display(df_group_type)

### EDA Analysis: Pivot Tables

In [0]:
# Create a pivot table for average orbital parameters by Object Type
grouped_pivot = df.groupby(['ObjectType', 'OrbitState'])[['OrbitalPeriodMin', 'MaxAltitudeKM']].mean().reset_index()
grouped_pivot_table = grouped_pivot.pivot(index='ObjectType', columns='OrbitState', values='MaxAltitudeKM')

# Display the pivot table
display(grouped_pivot_table)

### EDA Analysis: Correlation Matrix
*Raw correlation coefficients between numerical variables*

In [0]:
# 1. Select only the numerical columns
numeric_df = df.select_dtypes(include=['float64', 'int64'])

# 2. Calculate the correlation matrix
correlation_matrix = numeric_df.corr()

# 3. Display the matrix
correlation_matrix

### EDA Analysis: Pearson Coefficient and P-Value

In [0]:
from scipy import stats

# Calculate the Pearson Coefficient and p-value between Max Altitude and Orbital Period
# (Make sure to drop any potential missing/null values just for this calculation)
clean_data = df.dropna(subset=['MaxAltitudeKM', 'OrbitalPeriodMin'])

pearson_coef, p_value = stats.pearsonr(clean_data['MaxAltitudeKM'], clean_data['OrbitalPeriodMin'])

print("Pearson Correlation Coefficient:", pearson_coef)
print("P-value:", p_value)

## 8. Data Visualization

*Visual exploration of space debris data through charts, plots, and graphs*

### Visualization: Scatter Plot

In [0]:
import matplotlib.pyplot as plt

# Create the scatter plot
plt.figure(figsize=(10, 6)) # Makes the plot a bit larger and easier to read
plt.scatter(df['MinAltitudeKM'], df['MaxAltitudeKM'], alpha=0.5) # alpha makes the dots slightly transparent to see overlapping data

# Add labels and a title
plt.title('Satellite Orbits: Perigee vs. Apogee')
plt.xlabel('Perigee (Lowest Altitude in km)')
plt.ylabel('Apogee (Highest Altitude in km)')

# Show the plot
plt.show()

### Visualization: Regression Plot

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# Create the regression plot
# We use scatter_kws to make the dots transparent, and line_kws to make the trendline red
sns.regplot(x='MinAltitudeKM', y='MaxAltitudeKM', data=df, 
            scatter_kws={'alpha':0.3}, line_kws={'color':'red'})

plt.title('Regression Plot: Perigee vs. Apogee')
plt.xlabel('Perigee (Lowest Altitude in km)')
plt.ylabel('Apogee (Highest Altitude in km)')

plt.show()

### Visualization: Box Plot

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

# Create the box plot
sns.boxplot(x='ObjectType', y='OrbitalPeriodMin', data=df)

plt.title('Box Plot: Orbital Period Distribution by Object Type')
plt.xlabel('Object Type')
plt.ylabel('Orbital Period (Minutes)')

plt.show()

### Visualization: Heatmap (Pseudocolor Plot)

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

# Create the pseudocolor plot using our pivot table data
plt.pcolor(grouped_pivot_table.fillna(0), cmap='RdBu')

# Add labels and a color bar for reference
plt.title('Pseudocolor Plot of Object Types vs Altitudes')
plt.xlabel('Max Altitude / Columns')
plt.ylabel('Object Type / Rows')
plt.colorbar()

plt.show()

### Visualization: Histograms - Numerical Distributions

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Create histograms for key numerical variables
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Filter out sentinel values (-1) for better visualization
df_filtered = df[df['OrbitalPeriodMin'] >= 0]

# Orbital Period
axes[0, 0].hist(df_filtered['OrbitalPeriodMin'], bins=50, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Orbital Period')
axes[0, 0].set_xlabel('Orbital Period (Minutes)')
axes[0, 0].set_ylabel('Frequency')

# Inclination
axes[0, 1].hist(df_filtered[df_filtered['InclinationDegrees'] >= 0]['InclinationDegrees'], bins=50, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Distribution of Inclination')
axes[0, 1].set_xlabel('Inclination (Degrees)')
axes[0, 1].set_ylabel('Frequency')

# Max Altitude
axes[0, 2].hist(df_filtered[df_filtered['MaxAltitudeKM'] >= 0]['MaxAltitudeKM'], bins=50, color='lightgreen', edgecolor='black')
axes[0, 2].set_title('Distribution of Max Altitude')
axes[0, 2].set_xlabel('Max Altitude (km)')
axes[0, 2].set_ylabel('Frequency')

# Min Altitude
axes[1, 0].hist(df_filtered[df_filtered['MinAltitudeKM'] >= 0]['MinAltitudeKM'], bins=50, color='orange', edgecolor='black')
axes[1, 0].set_title('Distribution of Min Altitude')
axes[1, 0].set_xlabel('Min Altitude (km)')
axes[1, 0].set_ylabel('Frequency')

# Radar Size (log scale for better visibility)
radar_data = df[df['RadarSizeSQM'] > 0]['RadarSizeSQM']
axes[1, 1].hist(np.log10(radar_data), bins=50, color='purple', edgecolor='black')
axes[1, 1].set_title('Distribution of Radar Size (log scale)')
axes[1, 1].set_xlabel('Log10(Radar Size m²)')
axes[1, 1].set_ylabel('Frequency')

# Catalog ID over time
axes[1, 2].hist(df['CatalogID'], bins=50, color='pink', edgecolor='black')
axes[1, 2].set_title('Distribution of Catalog IDs')
axes[1, 2].set_xlabel('Catalog ID')
axes[1, 2].set_ylabel('Frequency')

plt.suptitle('Numerical Variable Distributions', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

### Visualization: Count Plots - Categorical Distributions

In [0]:
# Creating count plots for categorical variables
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Object Type
sns.countplot(data=df, x='ObjectType', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Count by Object Type', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Object Type')
axes[0, 0].set_ylabel('Count')
for container in axes[0, 0].containers:
    axes[0, 0].bar_label(container)

# Operational Status
sns.countplot(data=df, x='OperationalStatus', ax=axes[0, 1], palette='Set3')
axes[0, 1].set_title('Count by Operational Status', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Operational Status')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=45)
for container in axes[0, 1].containers:
    axes[0, 1].bar_label(container)

# Top 10 Owners
top_owners = df['Owner'].value_counts().head(10)
sns.barplot(x=top_owners.values, y=top_owners.index, ax=axes[1, 0], palette='viridis')
axes[1, 0].set_title('Top 10 Owners/Countries', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_ylabel('Owner')
for container in axes[1, 0].containers:
    axes[1, 0].bar_label(container)

# Orbit State
sns.countplot(data=df, x='OrbitState', ax=axes[1, 1], palette='coolwarm')
axes[1, 1].set_title('Count by Orbit State', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Orbit State')
axes[1, 1].set_ylabel('Count')
for container in axes[1, 1].containers:
    axes[1, 1].bar_label(container)

plt.suptitle('Categorical Variable Distributions', fontsize=16, y=0.995)
plt.tight_layout()
plt.show()

### Visualization: Time Series - Launch Trends Over Time

In [0]:
# Extract year from LaunchDate
df['LaunchYear'] = pd.to_datetime(df['LaunchDate']).dt.year

# Count launches per year
launches_per_year = df.groupby('LaunchYear').size()

# Create figure with subplots
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Overall launch trend
axes[0].plot(launches_per_year.index, launches_per_year.values, linewidth=2, color='darkblue')
axes[0].fill_between(launches_per_year.index, launches_per_year.values, alpha=0.3, color='skyblue')
axes[0].set_title('Total Space Object Launches Over Time', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Launches')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=2020, color='red', linestyle='--', alpha=0.5, label='2020 (Reference)')
axes[0].legend()

# Launch trend by Object Type
launches_by_type = df.groupby(['LaunchYear', 'ObjectType']).size().unstack(fill_value=0)
for obj_type in launches_by_type.columns:
    axes[1].plot(launches_by_type.index, launches_by_type[obj_type], linewidth=2, label=obj_type, marker='o', markersize=3)

axes[1].set_title('Space Object Launches by Type Over Time', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Launches')
axes[1].legend(title='Object Type')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n=== Launch Statistics ===")
print(f"Total launches: {len(df):,}")
print(f"Peak year: {launches_per_year.idxmax()} with {launches_per_year.max():,} launches")
print(f"First launch: {df['LaunchYear'].min()}")
print(f"Latest launch: {df['LaunchYear'].max()}")

### Visualization: Active vs Decayed Objects

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Create decay status flag
df['IsDecayed'] = df['DecayDate'].notna().astype(int)
df['IsActive'] = df['DecayDate'].isna().astype(int)

# Overall statistics
print("=== Space Object Status Overview ===")
print(f"Total objects in catalog: {len(df):,}")
print(f"\nObjects still in orbit: {df['IsActive'].sum():,} ({df['IsActive'].mean()*100:.1f}%)")
print(f"Objects decayed/re-entered: {df['IsDecayed'].sum():,} ({df['IsDecayed'].mean()*100:.1f}%)")

# Breakdown by Object Type
print("\n=== Status by Object Type ===")
status_by_type = df.groupby('ObjectType').agg({
    'IsActive': 'sum',
    'IsDecayed': 'sum'
}).rename(columns={'IsActive': 'Still in Orbit', 'IsDecayed': 'Decayed'})
print(status_by_type)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart - Overall status
status_counts = [df['IsActive'].sum(), df['IsDecayed'].sum()]
status_labels = ['Still in Orbit', 'Decayed/Re-entered']
colors = ['#2ecc71', '#e74c3c']
explode = (0.05, 0)

axes[0].pie(status_counts, labels=status_labels, autopct='%1.1f%%', startangle=90, 
            colors=colors, explode=explode, shadow=True, textprops={'fontsize': 12})
axes[0].set_title('Current Status of Space Objects', fontsize=14, fontweight='bold')

# Stacked bar chart - Status by type
status_by_type.plot(kind='bar', stacked=True, ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Object Status by Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Object Type')
axes[1].set_ylabel('Count')
axes[1].legend(title='Status')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Visualization: Decay Patterns Over Time

In [0]:


# Extract year from DecayDate for decayed objects
df_decayed = df[df['DecayDate'].notna()].copy()
df_decayed['DecayYear'] = pd.to_datetime(df_decayed['DecayDate']).dt.year

# Count decays per year
decays_per_year = df_decayed.groupby('DecayYear').size()

# Create visualization
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Overall decay trend
axes[0].bar(decays_per_year.index, decays_per_year.values, color='coral', edgecolor='darkred', alpha=0.7)
axes[0].set_title('Object Re-entries (Decays) Over Time', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Re-entries')
axes[0].grid(True, alpha=0.3, axis='y')

# Decay trend by Object Type
decays_by_type = df_decayed.groupby(['DecayYear', 'ObjectType']).size().unstack(fill_value=0)
for obj_type in decays_by_type.columns:
    axes[1].plot(decays_by_type.index, decays_by_type[obj_type], linewidth=2, label=obj_type, marker='o', markersize=4)

axes[1].set_title('Object Re-entries by Type Over Time', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Re-entries')
axes[1].legend(title='Object Type')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n=== Decay Statistics ===")
print(f"Total decayed objects: {len(df_decayed):,}")
print(f"Peak decay year: {decays_per_year.idxmax()} with {decays_per_year.max():,} re-entries")
print(f"First recorded decay: {df_decayed['DecayYear'].min()}")
print(f"Latest recorded decay: {df_decayed['DecayYear'].max()}")

### Visualization: Orbital Altitude Analysis

In [0]:
# Filter valid altitude data (exclude sentinel -1 values and extreme outliers)
df_orbits = df[(df['MaxAltitudeKM'] >= 0) & (df['MinAltitudeKM'] >= 0) & 
               (df['MaxAltitudeKM'] < 50000)].copy()

# Calculate average orbital altitude
df_orbits['AvgAltitudeKM'] = (df_orbits['MaxAltitudeKM'] + df_orbits['MinAltitudeKM']) / 2

# Define orbital regions
def classify_orbit(altitude):
    if altitude < 2000:
        return 'LEO (Low Earth Orbit)'
    elif altitude < 35586:
        return 'MEO (Medium Earth Orbit)'
    elif 35586 <= altitude <= 35986:
        return 'GEO (Geostationary Orbit)'
    else:
        return 'HEO (High Earth Orbit)'

df_orbits['OrbitRegion'] = df_orbits['AvgAltitudeKM'].apply(classify_orbit)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Distribution of average altitude
axes[0, 0].hist(df_orbits['AvgAltitudeKM'], bins=100, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=2000, color='red', linestyle='--', linewidth=2, label='LEO/MEO boundary (2000 km)')
axes[0, 0].axvline(x=35786, color='green', linestyle='--', linewidth=2, label='GEO altitude (~35,786 km)')
axes[0, 0].set_title('Distribution of Average Orbital Altitude', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Average Altitude (km)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()
axes[0, 0].set_xlim(0, 50000)

# Objects by orbital region
orbit_counts = df_orbits['OrbitRegion'].value_counts()
orbit_counts.plot(kind='bar', ax=axes[0, 1], color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'], edgecolor='black')
axes[0, 1].set_title('Objects by Orbital Region', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Orbital Region')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=45)
for container in axes[0, 1].containers:
    axes[0, 1].bar_label(container)

# Altitude by Object Type
df_orbits.boxplot(column='AvgAltitudeKM', by='ObjectType', ax=axes[1, 0])
axes[1, 0].set_title('Altitude Distribution by Object Type', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Object Type')
axes[1, 0].set_ylabel('Average Altitude (km)')
plt.sca(axes[1, 0])
plt.xticks(rotation=0)
axes[1, 0].get_figure().suptitle('')  # Remove default title

# Eccentricity analysis (orbital shape)
df_orbits['Eccentricity'] = (df_orbits['MaxAltitudeKM'] - df_orbits['MinAltitudeKM']) / (df_orbits['MaxAltitudeKM'] + df_orbits['MinAltitudeKM'])
axes[1, 1].hist(df_orbits['Eccentricity'], bins=50, color='purple', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Orbital Eccentricity Distribution', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Eccentricity (0 = circular, 1 = highly elliptical)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].axvline(x=0.1, color='red', linestyle='--', label='Moderate eccentricity (0.1)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print("=== Orbital Altitude Statistics ===")
print(f"\nOrbital Region Distribution:")
for region, count in orbit_counts.items():
    print(f"  {region}: {count:,} ({count/len(df_orbits)*100:.1f}%)")

print(f"\nAltitude Statistics (km):")
print(f"  Mean: {df_orbits['AvgAltitudeKM'].mean():.2f}")
print(f"  Median: {df_orbits['AvgAltitudeKM'].median():.2f}")
print(f"  Min: {df_orbits['AvgAltitudeKM'].min():.2f}")
print(f"  Max: {df_orbits['AvgAltitudeKM'].max():.2f}")

## Final Verification

In [0]:
# Final verification before Power BI export
print("="*60)
print("POWER BI EXPORT VERIFICATION")
print("="*60)

print(f"\n Dataset Shape: {df.shape}")
print(f"   - Rows: {len(df):,}")
print(f"   - Columns: {len(df.columns)}")

print(f"\n Key Features for Power BI:")
print(f"\n   Binary Flags (for measures & filters):")
for col in ['IsDecayed', 'IsActive', 'IsDebris', 'IsRocketBody']:
    if col in df.columns:
        print(f"      ✓ {col}: {df[col].sum():,} objects")

print(f"\n   Categorical Features (for slicers):")
for col in ['OrbitClass', 'RadarSize', 'LaunchEra']:
    if col in df.columns:
        print(f"      ✓ {col}: {df[col].nunique()} categories")
        print(f"        {df[col].value_counts().to_dict()}")

print(f"\n   Engineered Time Features:")
if 'LaunchYear' in df.columns:
    print(f"      ✓ LaunchYear: {df['LaunchYear'].min()} to {df['LaunchYear'].max()}")
if 'DecayYear' in df.columns:
    print(f"      ✓ DecayYear: available for time series")
if 'OrbitRegion' in df.columns:
    print(f"      ✓ OrbitRegion: {df['OrbitRegion'].nunique()} regions")

print(f"\n Export Location:")
export_path = '/Workspace/Users/guruvendra47@gmail.com/space-debris-project/space_debris_cleaned.csv'
print(f"   {export_path}")

print(f"\n Status: READY FOR POWER BI IMPORT")
print("="*60)

In [0]:
df.to_csv("/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/raw/space_debris_cleaned.csv", index=False)
